In [ ]:
# =====================================================================
# Pareto Front — NSGA-II Multi-Objective Optimization
# Objectives:  f1 = area (minimize),  f2 = grid outage (minimize)
# Channel model, grid, weights — all unchanged from grid_search.ipynb
# =====================================================================

import torch
import numpy as np
import math
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.termination import get_termination
from pymoo.optimize import minimize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ==========================================
# 1. System Parameters (unchanged)
# ==========================================
room_x, room_y, room_z_max = 10.0, 10.0, 3.0
x_BS, y_BS, z_BS = 10.0, -100.0, -10.0
E = 8; d_B = 0.075; lambda_val = 0.075; L1 = 2
d_ref = abs(y_BS) * 1.5; P_BS_dBm = 40.0; R_th = 0.2
N0_dBm_Hz = -174.0; B = 20e6; p_m = 1.0 / 5.0; n_users = 5
P_BS = 10**(P_BS_dBm / 10.0) * 1e-3
N0 = 10**(N0_dBm_Hz / 10.0) * 1e-3 * B

# Variable bounds (aligned with MATLAB lb/ub)
x_min, x_max = 0.2, 9.8
z_min, z_max = 0.2, 2.8
L_min, L_max = 0.1, 9.8
W_min, W_max = 0.1, 2.8   # Lz → W for clarity in Pareto context

# ==========================================
# 2. RWP Generator (for KDE weights)
# ==========================================
def gen_rwp_traj(sim_time,dt=10,nu=5,rx=10.0,ry=10.0,rng=None):
    if rng is None: rng=np.random
    rs=[0.0,rx,0.0,ry]; hc=np.array([rx/2,ry/2]); hr=rx/3.0
    p_t=0.6;p_s=0.3;tau_h=1.5;tau_n=0.1;v_h=0.2;v_n=1.0;g_h=0.6;g_n=0.2
    ts=int(sim_time/dt)
    def gt(p):
        t=hc+(rng.rand(2)-0.5)*2*hr if rng.rand()<g_h else np.array([rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])])
        t[0]=np.clip(t[0],rs[0],rs[1]);t[1]=np.clip(t[1],rs[2],rs[3]);return t
    def ta(r):return tau_h if r=='hot' else tau_n
    def sp(r):return v_h if r=='hot' else v_n
    uh=1.5+0.5*rng.rand(nu);pos=np.zeros((nu,ts,2))
    up=np.zeros((nu,2));ur=[None]*nu;us=[None]*nu;ut_=np.zeros((nu,2));ud=np.zeros((nu,2));usp=np.zeros(nu);uprem=np.zeros(nu)
    for i in range(nu):
        up[i]=[rs[0]+rng.rand()*(rs[1]-rs[0]),rs[2]+rng.rand()*(rs[3]-rs[2])];d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
        if rng.rand()<p_t:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
        else:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
        pos[i,0,:]=up[i]
    for step in range(1,ts):
        for i in range(nu):
            if us[i]=='pause':
                uprem[i]-=dt;pos[i,step,:]=up[i]
                if uprem[i]<=0:us[i]='transfer';ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
            else:
                md=usp[i]*dt;pd=ut_[i]-up[i]
                if np.linalg.norm(pd)<=md:
                    up[i]=ut_[i].copy();d2h=np.linalg.norm(up[i]-hc);ur[i]='hot' if d2h<=hr else 'normal'
                    if rng.rand()<p_s:us[i]='pause';uprem[i]=rng.exponential(ta(ur[i]))
                    else:ut_[i]=gt(up[i]);dv=ut_[i]-up[i];ud[i]=dv/np.linalg.norm(dv);usp[i]=sp(ur[i])
                else:up[i]=up[i]+ud[i]*md
                pos[i,step,:]=up[i]
    pts=np.zeros((nu*ts,3));idx=0
    for u in range(nu):
        for s in range(ts):pts[idx]=[pos[u,s,0],pos[u,s,1],uh[u]];idx+=1
    return pts

# ==========================================
# 3. Multi-Height Grid + KDE Weights
# ==========================================
GRID_RES = 200
Z_HEIGHTS = [1.5, 1.625, 1.75, 1.875, 2.0]
N_Z = len(Z_HEIGHTS)
x_vals = torch.linspace(0, room_x, GRID_RES, device=device)
y_vals = torch.linspace(0, room_y, GRID_RES, device=device)
Xg, Yg = torch.meshgrid(x_vals, y_vals, indexing='ij')
xy_flat = torch.stack([Xg.flatten(), Yg.flatten()], dim=1)
gl_list = []
for zh in Z_HEIGHTS:
    gl_list.append(torch.stack([Xg.flatten(), Yg.flatten(),
                                 torch.full_like(Xg.flatten(), zh)], dim=1))
grid_locs = torch.cat(gl_list, dim=0)
N_GRID = grid_locs.shape[0]
print(f'Grid: {GRID_RES}x{GRID_RES} = {N_GRID} points')

# ==========================================
# 3b. KDE Weights from 100ks RWP
# ==========================================
print('Building KDE weights (100,000s RWP)...')
np.random.seed(777)
emp_traj = gen_rwp_traj(sim_time=100000, dt=10)
emp_xy = emp_traj[:, :2].T
kde = gaussian_kde(emp_xy, bw_method='scott')
grid_xy_np = xy_flat.cpu().numpy().T
w_xy = kde(grid_xy_np).flatten()
w_xy = w_xy / w_xy.sum()
w_full = np.tile(w_xy, N_Z)
grid_weights = torch.tensor(w_full / w_full.sum(), dtype=torch.float32, device=device)
print(f'  Grid: {N_Z} heights x {GRID_RES}x{GRID_RES} = {N_GRID} pts, hotspot/edge = {grid_weights.max().item()/grid_weights.min().item():.1f}x')

# ==========================================
# 4. NLoS Params (fixed seed, one-time)
# ==========================================
np.random.seed(42)
_nlos_psi = torch.tensor(2*np.pi*np.random.rand(N_GRID), dtype=torch.float32, device=device)
_nlos_tt  = torch.tensor(-np.pi+2*np.pi*np.random.rand(N_GRID), dtype=torch.float32, device=device)
_nlos_eta = torch.tensor(10**((-15+5*np.random.rand(N_GRID))/10), dtype=torch.float32, device=device)

# ==========================================
# 5. Channel Model (static tensors pre-created)
# ==========================================
_n_ant = torch.arange(E, dtype=torch.float32, device=device)
_dy_BS = torch.tensor(0.0 - y_BS, device=device)
_v1m_c = lambda_val / (4.0 * math.pi)
_v1p_c = -(2.0 * math.pi / lambda_val)
_ones_E = 1.0 / math.sqrt(E)

def equivalent_farfield_channel_pytorch(window_params, locs):
    if not isinstance(window_params, torch.Tensor):
        window_params = torch.tensor(window_params, dtype=torch.float32, device=device)
    xc, zc, Lx, Lz = window_params[0], window_params[1], window_params[2], window_params[3]
    xu, yu, zu = locs[:,0], locs[:,1], locs[:,2]

    dx_BS=xc-x_BS; dy_BS=_dy_BS; dz_BS=zc-z_BS
    R_BW=torch.sqrt(dx_BS**2+dy_BS**2+dz_BS**2)
    theta_BW=torch.atan2(dy_BS,dx_BS); phi_BW=torch.acos(dz_BS/R_BW)
    k_tx=torch.sin(phi_BW)*torch.cos(theta_BW); k_tz=torch.cos(phi_BW)

    x1,z1=xc-Lx/2,zc-Lz/2; x2,z2=xc-Lx/2,zc+Lz/2
    x3,z3=xc+Lx/2,zc-Lz/2; x4,z4=xc+Lx/2,zc+Lz/2

    def ray_dir(xs,zs):
        dx=xs-x_BS; dy=_dy_BS; dz=zs-z_BS
        L=torch.sqrt(dx**2+dy**2+dz**2); return dx/L,dy/L,dz/L
    ux1,_,uz1=ray_dir(x1,z1); ux2,_,uz2=ray_dir(x2,z2)
    ux3,_,uz3=ray_dir(x3,z3); ux4,_,uz4=ray_dir(x4,z4)

    dx_WU=xu-x_BS; dy_WU=yu-y_BS; dz_WU=zu-z_BS
    L_USER=torch.sqrt(dx_WU**2+dy_WU**2+dz_WU**2)
    ux_user=dx_WU/L_USER; uz_user=dz_WU/L_USER

    ux_min=torch.min(torch.stack([ux1,ux2,ux3,ux4]))
    ux_max=torch.max(torch.stack([ux1,ux2,ux3,ux4]))
    uz_min=torch.min(torch.stack([uz1,uz2,uz3,uz4]))
    uz_max=torch.max(torch.stack([uz1,uz2,uz3,uz4]))

    beta=1000.0
    inx=torch.sigmoid(beta*(ux_user-ux_min))*torch.sigmoid(beta*(ux_max-ux_user))
    inz=torch.sigmoid(beta*(uz_user-uz_min))*torch.sigmoid(beta*(uz_max-uz_user))
    illum=inx*inz

    dx_WU2=xu-xc; dy_WU2=yu; dz_WU2=zu-zc
    R_WU=torch.sqrt(dx_WU2**2+dy_WU2**2+dz_WU2**2)
    t1=dx_WU2/R_WU; t2=dz_WU2/R_WU
    ax=(1.0-illum)*(k_tx-t1); az=(1.0-illum)*(k_tz-t2)
    sincx=torch.sinc((math.pi/lambda_val)*Lx*ax)
    sincz=torch.sinc((math.pi/lambda_val)*Lz*az)

    # Path 1 (LoS) — using pre-created globals
    ph1=(2.0*math.pi/lambda_val)*d_B*_n_ant*torch.sin(theta_BW)
    a1=_ones_E*torch.complex(torch.cos(ph1),torch.sin(ph1))
    v1m=_v1m_c/R_BW
    v1p=_v1p_c*R_BW
    v1=torch.complex(v1m*torch.cos(v1p),v1m*torch.sin(v1p))
    H1=v1*a1.conj()

    # Path 2 (NLoS)
    ph2=(2.0*math.pi/lambda_val)*d_B*n_ant.unsqueeze(0)*torch.sin(_nlos_tt).unsqueeze(1)
    a2=(1.0/math.sqrt(E))*torch.complex(torch.cos(ph2),torch.sin(ph2))
    v2m=_nlos_eta*(lambda_val/(4.0*math.pi*d_ref))
    v2=torch.complex(v2m*torch.cos(_nlos_psi),v2m*torch.sin(_nlos_psi))
    H2=v2.unsqueeze(1)*a2.conj()

    H_total=math.sqrt(E/L1)*(H1.unsqueeze(0)+H2)
    fm=(Lx*Lz*sincx*sincz)/(lambda_val*R_WU)
    fp=torch.tensor(-(2.0*math.pi/lambda_val),device=device)*(k_tx*xc+k_tz*zc)-(math.pi/2.0)
    factor=torch.complex(fm*torch.cos(fp),fm*torch.sin(fp))
    return H_total*factor.unsqueeze(1)

# ==========================================
# 6. Grid Outage Evaluator (deterministic, returns float)
# ==========================================
def compute_grid_outage(xc, zc, Lx, Lz):
    """Single forward pass: window params → weighted outage (Python float)"""
    theta = torch.tensor([xc, zc, Lx, Lz], dtype=torch.float32, device=device)
    H_eq = equivalent_farfield_channel_pytorch(theta, grid_locs)
    H_w = torch.sum(H_eq, dim=1)/math.sqrt(E)
    dp = (torch.abs(H_w)**2)*p_m*P_BS
    intf = (n_users-1)*dp
    sinr = dp/(intf+N0)
    # Self-blockage (Ref C.3): geometric body attenuation
    # Self-blockage (Ref C.3): E[Se] over 4 random facing directions
    r_proj_sq = (grid_locs[:,0]-xc)**2 + (grid_locs[:,2]-zc)**2
    r_sq = r_proj_sq + grid_locs[:,1]**2
    r_norm = torch.sqrt(r_sq + 1e-12)
    alpha_body = math.pi/3.0
    Se_sum = torch.zeros_like(sinr)
    for ang in [0.0, math.pi/2, math.pi, 3*math.pi/2]:
        dot = torch.abs((grid_locs[:,0]-xc)*math.cos(ang) + grid_locs[:,1]*math.sin(ang))
        cos_phi = dot / r_norm
        phi = torch.acos(torch.clamp(cos_phi,0,1))
        Se = (math.pi - torch.clamp(2*alpha_body - phi, min=0)) / math.pi
        Se_sum = Se_sum + torch.clamp(Se, 1.0/3.0, 1.0)
    Se = Se_sum / 4.0
    sinr = Se*dp/(Se*intf+N0)
    rate = torch.log2(1.0+sinr)
    outage_bits = (rate < R_th).float()
    return float(torch.sum(outage_bits*grid_weights).item())

# Quick sanity check
print(f'Sanity: [5,1.5,0.3,0.3] → outage={compute_grid_outage(5.0,1.5,0.3,0.3)*100:.2f}%\n')

# ==========================================
# 7. pymoo Problem Definition
#    f1 = area  (Lx * Lz)
#    f2 = grid outage (weighted spatial integral)
# ==========================================
class EMWindowProblem(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=4,
            n_obj=2,
            n_ieq_constr=4,
            xl=np.array([x_min, z_min, L_min, W_min]),
            xu=np.array([x_max, z_max, L_max, W_max])
        )

    def _evaluate(self, x, out, *args, **kwargs):
        xc, zc, Lx, Lz = x[0], x[1], x[2], x[3]

        # Inequality constraints: g_i(x) <= 0  for feasibility
        # Aligned with MATLAB nonlcon
        g1 = Lx/2 - xc               # -(xc-Lx/2) >= 0  →  Lx/2 - xc <= 0
        g2 = xc + Lx/2 - room_x       # xc+Lx/2 - room_x <= 0
        g3 = Lz/2 - zc               # -(zc-Lz/2) >= 0  →  Lz/2 - zc <= 0
        g4 = zc + Lz/2 - room_z_max   # zc+Lz/2 - room_z_max <= 0

        outage = compute_grid_outage(xc, zc, Lx, Lz)
        area = Lx * Lz

        out["F"] = [area, outage]
        out["G"] = [g1, g2, g3, g4]

# ==========================================
# 8. NSGA-II Configuration
# ==========================================
problem = EMWindowProblem()

algorithm = NSGA2(
    pop_size=300,
    n_offsprings=150,
    sampling=FloatRandomSampling(),
    crossover=SBX(prob=0.9, eta=15),
    mutation=PM(prob=0.25, eta=20),
    eliminate_duplicates=True
)

termination = get_termination("n_gen", 200)

print("="*60)
print(f" NSGA-II: pop=300, offspring=150, 200 generations")
print(f" Objectives: f1=area, f2=grid_outage")
print(f" Constraints: nonlcon boundary (4 inequalities)")
print(f" Total evaluations: ~{300*200} (all on GPU grid)")
print("="*60)

# ==========================================
# 9. Run Optimization
# ==========================================
res = minimize(problem, algorithm, termination, seed=42, verbose=True)

print(f"\nDone. Population size: {len(res.F)}")
print(f"Objective range: area [{res.F[:,0].min():.4f}, {res.F[:,0].max():.4f}]")
print(f"                 outage [{res.F[:,1].min()*100:.2f}%, {res.F[:,1].max()*100:.2f}%]")

# ==========================================
# 10. Pareto Front Visualization (res.F is already Pareto-optimal)
# ==========================================
sort_idx = np.argsort(res.F[:, 1])
F_sorted = res.F[sort_idx]

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(F_sorted[:,1]*100, F_sorted[:,0], 'o-', color='darkgreen',
        markersize=5, linewidth=1.5, label=f'Pareto Front ({len(F_sorted)} pts)')
ax.set_xlabel('Grid Outage [%]', fontsize=12)
ax.set_ylabel('Window Area [m²]', fontsize=12)
ax.set_title(f'Pareto Front: NSGA-II (pop=300, gen=200)', fontsize=13)
ax.axvline(x=10, color='red', linestyle='--', alpha=0.5, label='10% outage')
ax.grid(True, alpha=0.3)

feasible = res.F[:,1] <= 0.10
if feasible.any():
    best_idx = np.where(feasible)[0][np.argmin(res.F[feasible, 0])]
    bac,boc = res.F[best_idx,0], res.F[best_idx,1]
    ax.plot(boc*100, bac, 'r*', markersize=14, markeredgewidth=2,
            label=f'Knee: {bac:.4f} m² @ {boc*100:.1f}%')
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('pareto_front.png', dpi=150); plt.show()

# ==========================================
# 11. Knee Report
# ==========================================
if feasible.any():
    bx = res.X[best_idx]
    print("\n" + "="*60)
    print("Knee Solution (smallest area with outage <= 10%)")
    print("="*60)
    print(f"  xc={bx[0]:.3f} m,  zc={bx[1]:.3f} m")
    print(f"  Lx={bx[2]:.3f} m,  Lz={bx[3]:.3f} m")
    print(f"  Area = {bac:.4f} m²")
    print(f"  Grid Outage = {boc*100:.2f}%")
    print("="*60)

print("\nPareto front analysis complete.")